# System Metrics Monitoring

When running latency benchmarks — especially load tests with high concurrency — it's important to know whether the **client machine itself** is a bottleneck. High CPU usage, memory pressure, or network saturation on the client side can skew latency measurements without any visible errors.

LLMeter's `SystemMetricsMonitor` callback tracks CPU, memory, and network I/O during benchmark runs using [psutil](https://github.com/giampaolo/psutil). This notebook demonstrates how to:

1. Attach system metrics monitoring to a Runner
2. Inspect aggregated statistics from the Result
3. Plot time-series of system resources within a run
4. Use LoadTest to correlate system resources with concurrency
5. Detect client-side bottlenecks

## Setup

Install LLMeter with the `system-metrics` extra (which brings in `psutil`) and `plotting` for visualization:

In [ ]:
%pip install "llmeter[system-metrics,plotting]<1"

In [ ]:
from llmeter.callbacks.system_metrics import SystemMetricsMonitor
from llmeter.endpoints.bedrock import BedrockConverseStream
from llmeter.experiments import LoadTest
from llmeter.runner import Runner

This notebook assumes you have [configured AWS credentials](https://boto3.amazonaws.com/v1/documentation/api/latest/guide/credentials.html) with `bedrock:InvokeModel` and `bedrock:InvokeModelWithResponseStream` permissions, and that you've [enabled access](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html) to the model below.

In [ ]:
model_id = None  # <-- Set a Bedrock model ID, e.g. "us.anthropic.claude-3-5-haiku-20241022-v1:0"

if not model_id:
    raise ValueError("Please set a valid model ID above!")

In [ ]:
endpoint = BedrockConverseStream(model_id=model_id)

payload = endpoint.create_payload(
    "Write a short haiku about cloud computing.",
    max_tokens=100,
)

## Basic Usage

Create a `SystemMetricsMonitor` and attach it to a `Runner`. The monitor spawns a lightweight background thread that samples system metrics at the configured interval. After the run, aggregated statistics are contributed directly to `result.stats`.

In [ ]:
monitor = SystemMetricsMonitor(sample_interval=0.5, per_process=True)

runner = Runner(
    endpoint=endpoint,
    callbacks=[monitor],
    output_path="outputs/system-metrics",
)

result = await runner.run(payload=payload, clients=5, n_requests=10)

## Inspecting Results

System metrics are part of `result.stats` — the same interface used for latency and throughput statistics. No need to access the monitor object for aggregated values:

In [ ]:
# CPU usage during the run
print(f"CPU average: {result.stats['system_cpu_percent-average']:.1f}%")
print(f"CPU p90:     {result.stats['system_cpu_percent-p90']:.1f}%")
print(f"CPU p99:     {result.stats['system_cpu_percent-p99']:.1f}%")
print()

# Memory usage
print(f"Memory RSS average: {result.stats['system_memory_rss_mb-average']:.1f} MB")
print(f"Memory RSS peak:    {result.stats['system_memory_rss_mb-max']:.1f} MB")
print()

# Network I/O
print(f"Network sent:     {result.stats['system_net_bytes_sent_total']:,} bytes")
print(f"Network received: {result.stats['system_net_bytes_recv_total']:,} bytes")
print()

# How many samples were collected
print(f"Samples collected: {result.stats['system_samples_collected']}")

These stats persist across save/load — they're stored in `stats.json` alongside latency metrics:

In [ ]:
from llmeter.results import Result

if result.output_path:
    loaded = Result.load(result.output_path)
    print(f"Loaded CPU avg: {loaded.stats['system_cpu_percent-average']:.1f}%")
    print(f"Loaded RSS max: {loaded.stats['system_memory_rss_mb-max']:.1f} MB")

## Plotting Time-Series

The monitor provides a built-in `plot_samples()` method that visualizes how CPU, memory, and network evolved over the duration of the run:

In [ ]:
monitor.plot_samples()

## Correlating System Metrics with Concurrency

The most common use case for system monitoring is understanding whether client resources become a bottleneck as concurrency increases. Use `LoadTest` to sweep across concurrency levels, then `plot_results(extra_stats=...)` to visualize system metrics alongside the standard charts.

In [ ]:
load_test = LoadTest(
    endpoint=endpoint,
    payload=payload,
    sequence_of_clients=[1, 5, 10, 20],
    min_requests_per_client=5,
    callbacks=[monitor],
    output_path="outputs/system-metrics-load-test",
)

load_test_result = await load_test.run()

In [ ]:
load_test_result.plot_results(extra_stats={
    "system_cpu_percent-average": "CPU avg (%)",
    "system_cpu_percent-p90": "CPU p90 (%)",
    "system_memory_rss_mb-max": "RSS peak (MB)",
    "system_net_bytes_recv_per_second-average": "Net recv rate (bytes/s)",
})

## Detecting Client-Side Bottlenecks

After a load test, check these indicators across concurrency levels:

- **CPU p90 approaching 100%** — the client process can't keep up with scheduling requests.
- **Memory RSS growing significantly** — response data or asyncio tasks accumulating.
- **Network rates plateauing while latency increases** — possible network saturation.

If you observe these, consider running LLMeter on a more powerful instance or reducing concurrency.

In [ ]:
for clients in sorted(load_test_result.results.keys()):
    r = load_test_result.results[clients]
    cpu_p90 = r.stats["system_cpu_percent-p90"]
    rss_max = r.stats["system_memory_rss_mb-max"]
    status = "\u26a0\ufe0f  HIGH CPU" if cpu_p90 > 80 else "\u2705 OK"
    print(f"clients={clients:2d}  CPU p90={cpu_p90:5.1f}%  RSS max={rss_max:6.1f} MB  {status}")

## Raw Samples

For custom analysis beyond the built-in plots, you can access the raw samples from the monitor. Each sample is a dataclass with `timestamp`, `cpu_percent`, `memory_rss_mb`, `memory_vms_mb`, `net_bytes_sent`, and `net_bytes_recv`.

> **Note:** Raw samples live only on the monitor instance — they are not persisted in `stats.json`. Only the aggregated statistics survive save/load.

In [ ]:
# Iterate directly — no extra dependencies needed
t0 = monitor.samples[0].timestamp

print(f"{'Time (s)':>8}  {'CPU %':>6}  {'RSS MB':>7}  {'Net Recv':>12}")
print("-" * 40)
for s in monitor.samples[:10]:
    print(f"{s.timestamp - t0:8.2f}  {s.cpu_percent:6.1f}  {s.memory_rss_mb:7.1f}  {s.net_bytes_recv:>12,}")

In [ ]:
# If you prefer a DataFrame, samples are dataclasses — use asdict:
from dataclasses import asdict

import pandas as pd

df = pd.DataFrame([asdict(s) for s in monitor.samples])
df["time"] = df["timestamp"] - df["timestamp"].iloc[0]
df.head()

## System-Wide vs Per-Process Monitoring

By default, `SystemMetricsMonitor` tracks the current Python process only (`per_process=True`). Set `per_process=False` to monitor the entire machine — useful when other processes (e.g., a local model server) contribute to the workload.

> **Note:** Network I/O is always system-wide regardless of this setting, since `psutil` doesn't support per-process network counters on most platforms.

In [ ]:
system_wide_monitor = SystemMetricsMonitor(sample_interval=0.5, per_process=False)

system_runner = Runner(endpoint=endpoint, callbacks=[system_wide_monitor])
result_wide = await system_runner.run(payload=payload, clients=5, n_requests=5)

print(f"System-wide CPU avg:   {result_wide.stats['system_cpu_percent-average']:.1f}%")
print(f"System-wide Memory:    {result_wide.stats['system_memory_rss_mb-max']:.1f} MB (used)")
print(f"System-wide VMS:       {result_wide.stats['system_memory_vms_mb-max']:.1f} MB (total)")

## Summary

- **Aggregated stats** (CPU, memory, network) are part of `result.stats` — use the standard Result interface.
- **`monitor.plot_samples()`** gives you a time-series visualization of the last run.
- **`load_test_result.plot_results(extra_stats=...)`** plots system metrics vs concurrency alongside standard charts.
- **Raw samples** on `monitor.samples` are dataclasses you can iterate directly or convert with `dataclasses.asdict`.
- The monitor **resets between runs** — a single instance works across multiple `runner.run()` calls or LoadTest levels.
- Stats **persist** in `stats.json` across save/load; raw samples do not.